# General Powerful Scalable Graph Transformer (GPS)

Molecular Property Prediction on ZINC: Hybrid architecture combining local message passing with global full-attention transformers. This notebook implements the approach with `GPSLayer` inside a `K3GraphGPS` model, trained with the Adam optimizer, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `GPSLayer` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install git+http://github.com/anas-rz/k3-node/@main

# ==============================================================================
# Part 2: K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

import k3_node
from k3_node import layers as k3_layers

title = "General Powerful Scalable Graph Transformer (GraphGPS)"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. GraphGPS Model using GPSConv
class K3GraphGPS(keras.Model):
    def __init__(self, channels=64, num_layers=3, num_classes=2):
        super().__init__()
        self.node_emb = layers.Dense(channels)
        self.convs = []
        for _ in range(num_layers):
            local_gnn = k3_layers.GCNConv(channels, channels)
            self.convs.append(k3_layers.GPSConv(channels, local_gnn=local_gnn, heads=4))
        self.lin = layers.Dense(num_classes)

    def call(self, x, edge_index, batch=None):
        x = self.node_emb(x)
        for conv in self.convs:
            x = conv(x, edge_index, batch=batch)
        out = k3_layers.global_mean_pool(x, batch)
        return self.lin(out)

k3_model = K3GraphGPS(channels=32, num_layers=2, num_classes=2)

# 2. Forward pass verification
num_nodes = 20
dummy_x = keras.random.normal((num_nodes, 10))
dummy_edges = ops.convert_to_tensor([[0, 1], [1, 0]], dtype="int64")
dummy_batch = ops.zeros((num_nodes,), dtype="int64")

out = k3_model(dummy_x, dummy_edges, dummy_batch)
print(f"GraphGPS forward pass successful! Output shape: {out.shape}")

k3_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
)
print("GraphGPS model compiled successfully!")

print("\n✓ K3-Node GraphGPS execution completed successfully!")